<a href="https://colab.research.google.com/github/srikala2/23K81A05C4/blob/main/Global_Climate_Event_Economic_Impact_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install plotly scikit-learn

In [8]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

In [2]:
from google.colab import files
uploaded = files.upload()

Saving global_climate_events_economic_impact_2020_2025.csv to global_climate_events_economic_impact_2020_2025.csv


In [9]:
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
df.head()

,event_id,date,year,month,country,event_type,severity,duration_days,affected_population,deaths,injuries,economic_impact_million_usd,infrastructure_damage_score,response_time_hours,international_aid_million_usd,latitude,longitude,total_casualties,impact_per_capita,aid_percentage
0,EV01539,2020-01-01,2020,1,Japan,Tsunami,1,1,420956,0,2,0.01,4.9,11,0.0,85.4321,138.7206,2,0.02,0.0
1,EV02303,2020-01-01,2020,1,Qatar,Hurricane,1,4,3276,1,10,0.00,3.4,5,0.0,-32.0370,14.0111,11,0.00,0.0
2,EV01796,2020-01-02,2020,1,Canada,Drought,3,6,120382,0,9,0.10,8.9,10,0.0,78.4213,-112.7556,9,0.83,0.0
3,EV00175,2020-01-02,2020,1,Poland,Heatwave,6,16,185527,2,37,1.27,17.8,7,0.0,73.6564,115.0650,39,6.85,0.0
4,EV01115,2020-01-03,2020,1,UAE,Wildfire,4,16,176642,2,27,2.01,18.7,17,0.0,52.6458,101.5023,29,11.38,0.0


In [10]:
df = df.drop_duplicates()
df = df.fillna(0)

label_encoders = {}
for col in df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 20 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   event_id                       3000 non-null   int64  
 1   date                           3000 non-null   int64  
 2   year                           3000 non-null   int64  
 3   month                          3000 non-null   int64  
 4   country                        3000 non-null   int64  
 5   event_type                     3000 non-null   int64  
 6   severity                       3000 non-null   int64  
 7   duration_days                  3000 non-null   int64  
 8   affected_population            3000 non-null   int64  
 9   deaths                         3000 non-null   int64  
 10  injuries                       3000 non-null   int64  
 11  economic_impact_million_usd    3000 non-null   float64
 12  infrastructure_damage_score    3000 non-null   f

In [11]:
print(df.columns)


Index(['event_id', 'date', 'year', 'month', 'country', 'event_type',
       'severity', 'duration_days', 'affected_population', 'deaths',
       'injuries', 'economic_impact_million_usd',
       'infrastructure_damage_score', 'response_time_hours',
       'international_aid_million_usd', 'latitude', 'longitude',
       'total_casualties', 'impact_per_capita', 'aid_percentage'],
      dtype='object')


In [12]:
df['economic_impact_million_usd'] = pd.to_numeric(
    df['economic_impact_million_usd'], errors='coerce'
)
print(df['economic_impact_million_usd'].describe())
event_impact = df.groupby('event_type')['economic_impact_million_usd'].sum().reset_index()

print(event_impact.head())
print("Total rows after grouping:", event_impact.shape)

count    3000.000000
mean        1.944367
std        15.734990
min         0.000000
25%         0.010000
50%         0.090000
75%         0.530000
max       718.210000
Name: economic_impact_million_usd, dtype: float64
   event_type  economic_impact_million_usd
0           0                      1212.60
1           1                      1336.11
2           2                        27.54
3           3                       309.94
4           4                        78.21
Total rows after grouping: (12, 2)


In [13]:
print(df.shape)

(3000, 20)


In [14]:
import plotly.express as px
deaths_by_country = df.groupby('country')['deaths'].sum().reset_index()
deaths_by_country = deaths_by_country.sort_values(by='deaths', ascending=False)
fig = px.pie(
    deaths_by_country,
    values='deaths',
    names='country',
    title='Total Deaths by Country',
    template='plotly_white'
)

fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

In [15]:
import plotly.express as px
df['economic_impact_million_usd'] = pd.to_numeric(
    df['economic_impact_million_usd'], errors='coerce'
)
event_impact = df.groupby('event_type')[
    'economic_impact_million_usd'
].sum().reset_index()

print(event_impact.head())
fig1 = px.bar(
    event_impact,
    x='event_type',
    y='economic_impact_million_usd',
    title='Total Economic Impact by Climate Event Type',
    template='plotly_white'
)

fig1.show()

   event_type  economic_impact_million_usd
0           0                      1212.60
1           1                      1336.11
2           2                        27.54
3           3                       309.94
4           4                        78.21


In [16]:
import pandas as pd
import plotly.express as px
numeric_data = df.select_dtypes(include=['int64', 'float64'])
correlation_matrix = numeric_data.corr()
fig2 = px.imshow(
    correlation_matrix,
    text_auto=True,
    aspect="auto",
    color_continuous_scale='RdBu_r',
    title='Correlation Matrix of Climate Event Impact Variables'
)

fig2.update_layout(title_x=0.5)

fig2.show()


In [17]:
df['deaths'] = pd.to_numeric(df['deaths'], errors='coerce')
df['total_casualties'] = pd.to_numeric(df['total_casualties'], errors='coerce')

impact_summary = (
    df.groupby('event_type', as_index=False)
    .agg(
        total_deaths=('deaths', 'sum'),
        total_casualties=('total_casualties', 'sum')
    )
)

print(impact_summary.head())

fig3 = px.line(
    impact_summary,
    x='event_type',
    y='total_deaths',
    title='Total Deaths by Climate Event Type',
    labels={
        'event_type': 'Event Type',
        'total_deaths': 'Total Deaths'
    },
    template='plotly_dark'
)

fig3.update_layout(title_x=0.5)

fig3.show()


   event_type  total_deaths  total_casualties
0           0          1435             13858
1           1          1756             14850
2           2           451              4373
3           3          1688             13622
4           4          1380             13832


In [18]:
import plotly.express as px
df['economic_impact_million_usd'] = pd.to_numeric(
    df['economic_impact_million_usd'],
    errors='coerce'
)
annual_trend = (
    df.groupby('year', as_index=False)
    .agg(total_impact=('economic_impact_million_usd', 'sum'))
)

print(annual_trend)
fig4 = px.area(
    annual_trend,
    x='year',
    y='total_impact',
    title='Annual Economic Impact of Climate Events (2020–2025)',
    labels={
        'year': 'Year',
        'total_impact': 'Total Economic Impact (Million USD)'
    },
    template='plotly_dark'
)

fig4.update_layout(title_x=0.5)

fig4.show()

   year  total_impact
0  2020       1972.30
1  2021       1005.49
2  2022        879.02
3  2023        590.17
4  2024        850.61
5  2025        535.51


In [21]:
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

fig5 = px.scatter_3d(
    df,
    x='total_casualties',
    y='country',
    z='year',
    color='event_type',
    size='total_casualties',
    hover_data=['international_aid_million_usd', 'injuries'],
    opacity=0.8,
    template='plotly_dark'
)
fig5.update_layout(
    title='3D Visualization: Total Casualties vs Country vs Year',
    scene=dict(
        xaxis_title='Total Casualties',
        yaxis_title='Country',
        zaxis_title='Year',
        xaxis=dict(backgroundcolor="rgb(230, 230,230)"),
        yaxis=dict(backgroundcolor="rgb(230, 230,230)"),
        zaxis=dict(backgroundcolor="rgb(230, 230,230)"),
    ),
    margin=dict(l=0, r=0, b=0, t=50),
)
frames = [go.Frame(layout=dict(scene_camera=dict(eye=dict(x=2*np.cos(angle), y=2*np.sin(angle), z=1.2))))
          for angle in np.linspace(0, 2*np.pi, 60)]

fig5.update(frames=frames)
fig5.show()

In [22]:
fig6 = px.histogram(df, x='country', color='injuries', template="plotly",
                     title='Country-wise Imapct of Global Climate Events ')
fig6.show()

In [23]:
fig7 = px.scatter(df,
                  x='international_aid_million_usd',
                  y='total_casualties',
                  size='deaths',
                  color='total_casualties',
                  hover_data=['country','year'],
                  template="plotly_dark"
)
fig7.update_layout(
                  title='International_aid_million vs Total_casualties',
                  xaxis_title='International_aid_million_usd',
                  yaxis_title='total_casualties')
fig7.show()

In [24]:
pip install dash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 81.9 MB/s eta 0:00:00


In [ ]:
import dash
from dash import html   #to code html components
import dash.dcc as dcc
import plotly.graph_objects as go
import plotly.express as px
from dash.dependencies import Input, Output
from plotly.subplots import make_subplots
external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']
app = dash.Dash(__name__, external_stylesheets=external_stylesheets)
colors = {
    'background': '#111111',
    'text': '#7FDBFF'}
app.layout = html.Div(style={'backgroundColor': colors['background']}, children=[
    html.H1(
        children='"Global Climate Events & Economic impact"',
        style={
            'textAlign': 'center',
            'color': colors['text'] }   ),
html.Div(children='  - By Srikala', style={
        'textAlign': 'center',
        'color': colors['text']  } ),
    dcc.Graph(id='firstgraph1',figure=fig,style={'width':'90%','display':'inline-block','backgroundColor': colors['background']}),
    dcc.Graph(id='firstgraph2',figure=fig1,style={'width':'90%','display':'inline-block','backgroundColor': colors['background']}),
    dcc.Graph(id='firstgraph9',figure=fig2,style={'width':'90%','display':'inline-block','backgroundColor': colors['background']}),
    dcc.Graph(id='firstgraph3',figure=fig3,style={'width':'90%','display':'inline-block','backgroundColor': colors['background']}),
    dcc.Graph(id='firstgraph4',figure=fig4,style={'width':'90%','display':'inline-block','backgroundColor': colors['background']}),
    dcc.Graph(id='firstgraph5',figure=fig5,style={'width':'90%','display':'inline-block','backgroundColor': colors['background']}),
    dcc.Graph(id='firstgraph6',figure=fig6,style={'width':'90%','display':'inline-block','backgroundColor': colors['background']}),
    dcc.Graph(id='firstgraph7',figure=fig7,style={'width':'90%','display':'inline-block','backgroundColor': colors['background']}),

])


if __name__ == '__main__':
    app.run()

Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:8050
INFO:werkzeug:Press CTRL+C to quit
